# DrugAge → Knowledge Graph (KG) Builder

**Source:** [DrugAge](https://genomics.senescence.info/drugs/) database — two source files processed in parallel  



In [1]:
# ! wget https://genomics.senescence.info/drugs/dataset.zip 
# https://genomics.senescence.info/drugs/browse.php

---
## 0 · Configuration — edit ONLY these two lines

In [2]:
import os
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : root folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/drugage"

# ── Derived input paths (do not edit below this line) ────────────────────────
# PubChem CID-Synonym flat file (tab-separated, no header: col-0=CID, col-1=synonym)
PUBCHEM_SYN_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-Synonym-filtered")
# PubChem compound table with IUPAC names and SMILES (pickled DataFrame)
PUBCHEM_PKL_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.pkl")
# DrugAge source file 1 (original flat CSV)
DRUGAGE_F1_PATH   = os.path.join(BASE_PATH, "drugage/drugage.csv")
# DrugAge source file 2 (Browse export with richer metadata)
DRUGAGE_F2_PATH   = os.path.join(BASE_PATH, "drugage/DrugAge_Browse.csv")

# ── Derived output directories — one sub-folder per species ──────────────────
OUT_YEAST = os.path.join(OUT_PATH, "Yeast/")
OUT_CELE  = os.path.join(OUT_PATH, "Cele/")
OUT_DROSO = os.path.join(OUT_PATH, "Droso/")
OUT_MOUSE = os.path.join(OUT_PATH, "Mouse/")

# Intermediate directory — F1/F2 per-species files written here before merging
OUT_INTER = os.path.join(OUT_PATH, "drugage_intermediate/")

for d in [OUT_YEAST, OUT_CELE, OUT_DROSO, OUT_MOUSE, OUT_INTER]:
    os.makedirs(d, exist_ok=True)

# Species kept throughout the pipeline
SPECIES_LIST = [
    'Drosophila melanogaster', 'Mus musculus', 'Saccharomyces cerevisiae',
    'Caenorhabditis elegans', 'Danio rerio', 'Homo sapiens', 'Humans'
]

# Final merged output schema (lowercase — applied after column lowercasing)
REQUIRED_COLUMNS = [
    'head', 'relation', 'tail', 'head_type', 'tail_type',
    'relation_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'head_alt_name', 'species'
]

print("Paths configured successfully.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output root : {OUT_PATH}")

Paths configured successfully.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output root : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/drugage


---
## 1 · Load PubChem reference dictionaries

Three lookups built once here, shared by both F1 and F2 pipelines:

| Dictionary | Key | Value |
|---|---|---|
| `Pubchem_Syn_fil_dict` | Synonym string | PubChem CID |
| `Pubchem_IUPAC_CID_Dict` | PubChem CID | IUPAC name |
| `Pubchem_CID_Smile_Dict` | PubChem CID | SMILES string |

In [2]:
# ── 1a. Synonym → CID lookup ──────────────────────────────────────────────────
# CID-Synonym-filtered: col-0 = PubChem CID, col-1 = synonym string.
# Inverted so we can resolve a compound name to its CID.
Pubchem_Syn_fil = pd.read_csv(PUBCHEM_SYN_PATH, sep='\t', header=None)
Pubchem_Syn_fil_dict = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))  # {synonym: CID}

# Pre-build a lowercase version for case-insensitive lookups (used in both pipelines)
Pubchem_Syn_fil_dict_lower = {str(k).lower(): v for k, v in Pubchem_Syn_fil_dict.items()}

print(f"Loaded {len(Pubchem_Syn_fil_dict):,} PubChem synonyms")

Loaded 102,877,691 PubChem synonyms


In [3]:
# ── 1b. CID → IUPAC name and SMILES ──────────────────────────────────────────
# combined_df.pkl columns: PUBCHEM_COMPOUND_CID | PUBCHEM_IUPAC_NAME | PUBCHEM_SMILES
Pubchem = pd.read_pickle(PUBCHEM_PKL_PATH)

Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))  # {CID: IUPAC}
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))       # {CID: SMILES}

print(f"Loaded {len(Pubchem_IUPAC_CID_Dict):,} CID → IUPAC mappings")
print(f"Loaded {len(Pubchem_CID_Smile_Dict):,} CID → SMILES mappings")
Pubchem.head(3)

Loaded 119,177,440 CID → IUPAC mappings
Loaded 119,177,440 CID → SMILES mappings


,PUBCHEM_COMPOUND_CID,PUBCHEM_IUPAC_NAME,PUBCHEM_SMILES
0,56500001,"2-[1-(4,6-dimethylpyrimidin-2-yl)-3,5-dimethyl...",CC1=CC(=NC(=N1)N2C(=C(C(=N2)C)CC(=O)NC3=C(N=CC...
1,56500002,N-[2-(2-methoxyethoxy)pyridin-3-yl]-2-[[4-(4-m...,COCCOC1=C(C=CC=N1)NC(=O)CSC2=NN=CN2C3=CC=C(C=C...
2,56500003,N-[2-(2-methoxyethoxy)pyridin-3-yl]-1-phenyl-5...,COCCOC1=C(C=CC=N1)NC(=O)C2=NN(C(=N2)C3=CC=CS3)...


---
## 2 · File 1 (F1) Pipeline — `drugage.csv`



In [6]:
# ── 2a. Load F1 raw data ──────────────────────────────────────────────────────
file_drugage = pd.read_csv(DRUGAGE_F1_PATH)
print(f"F1 raw: {len(file_drugage):,} rows")
print(f"Lifespan change range: {file_drugage['avg_lifespan_change_percent'].min():.1f}% → {file_drugage['avg_lifespan_change_percent'].max():.1f}%")

# Drop only NaN — keep both S and NS
file_drugage = file_drugage[file_drugage['avg_lifespan_significance'].notna()]

file_drugage

F1 raw: 3,423 rows
Lifespan change range: -96.0% → 267.9%


,compound_name,species,strain,dosage,age_at_initiation,treatment_duration,avg_lifespan_change_percent,avg_lifespan_significance,max_lifespan_change_percent,max_lifespan_significance,gender,weight_change_percent,weight_change_significance,ITP,pubmed_id
52,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.0001 mM,NaN,NaN,2.30,NS,NaN,NaN,Male,NaN,NaN,No,98863
53,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.001 mM,NaN,NaN,1.40,NS,NaN,NaN,Male,NaN,NaN,No,98863
54,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.01 mM,NaN,NaN,-1.15,NS,NaN,NaN,Male,NaN,NaN,No,98863
55,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.1 mM,NaN,NaN,-0.85,NS,NaN,NaN,Male,NaN,NaN,No,98863
56,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.0001 mM,NaN,NaN,-2.30,NS,NaN,NaN,Male,NaN,NaN,No,98863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3418,Canagliflozin,Mus musculus,UM-HET3,180 ppm food,16 months,until death,14.00,S,4.0,S,Male,-9.0,S,Yes,38753230
3419,Alpha-ketoglutarate,Mus musculus,UM-HET3,20000 ppm food,18 months,until death,1.00,NS,-5.0,NS,Female,8.0,NS,Yes,38753230
3420,Alpha-ketoglutarate,Mus musculus,UM-HET3,20000 ppm food,18 months,until death,3.00,NS,-3.0,NS,Male,0.0,NS,Yes,38753230
3421,X203 (anti-IL-11),Mus musculus,C57BL/6 J,40 ppm bodyweight monthly intraperitoneal inje...,17.2 months,until death,25.00,S,NaN,NaN,Female,NaN,NaN,No,39020175


In [8]:
file_drugage['avg_lifespan_significance'].value_counts()

avg_lifespan_significance
S     2095
NS    1117
Name: count, dtype: int64

In [5]:
display(file_drugage['avg_lifespan_change_percent'].value_counts())

avg_lifespan_change_percent
 0.00     121
 10.00     55
 14.00     44
 3.00      43
 1.00      41
         ... 
-7.13       1
 4.97       1
-4.68       1
-1.29       1
-2.94       1
Name: count, Length: 1418, dtype: int64

In [48]:
# ── Step 1: Group by compound_name + species ──────────────────────────────────
def classify_compound(group):
    sig_rows = group[group['avg_lifespan_significance'] == 'S']
    
    # If NO significant rows at all → NotAssociated
    if len(sig_rows) == 0:
        group['Relation'] = 'NotAssociatedWith'
        return group
    
    # Has significant rows → assign direction per S row
    # NS rows within same compound → also NotAssociated
    def assign_direction(row):
        if row['avg_lifespan_significance'] != 'S':
            return 'NotAssociatedWith'
        pct = row['avg_lifespan_change_percent']
        if pct > 1:
            return 'PositivelyAssociatedWith'
        elif pct < -1:
            return 'NegativelyAssociatedWith'
        else:
            return 'NotAssociatedWith'
    
    group['Relation'] = group.apply(assign_direction, axis=1)
    return group

file_drugage = (
    file_drugage
    .groupby(['compound_name', 'species'], group_keys=False)
    .apply(classify_compound)
)

print("Tail distribution:")
print(file_drugage['Relation'].value_counts())
file_drugage

Tail distribution:
Relation
PositivelyAssociatedWith    1664
NotAssociatedWith           1151
NegativelyAssociatedWith     397
Name: count, dtype: int64


/tmp/ipykernel_2371907/1436497765.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(classify_compound)


,compound_name,species,strain,dosage,age_at_initiation,treatment_duration,avg_lifespan_change_percent,avg_lifespan_significance,max_lifespan_change_percent,max_lifespan_significance,gender,weight_change_percent,weight_change_significance,ITP,pubmed_id,Relation
52,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.0001 mM,NaN,NaN,2.30,NS,NaN,NaN,Male,NaN,NaN,No,98863,NotAssociatedWith
53,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.001 mM,NaN,NaN,1.40,NS,NaN,NaN,Male,NaN,NaN,No,98863,NotAssociatedWith
54,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.01 mM,NaN,NaN,-1.15,NS,NaN,NaN,Male,NaN,NaN,No,98863,NotAssociatedWith
55,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.1 mM,NaN,NaN,-0.85,NS,NaN,NaN,Male,NaN,NaN,No,98863,NotAssociatedWith
56,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.0001 mM,NaN,NaN,-2.30,NS,NaN,NaN,Male,NaN,NaN,No,98863,NotAssociatedWith
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3418,Canagliflozin,Mus musculus,UM-HET3,180 ppm food,16 months,until death,14.00,S,4.0,S,Male,-9.0,S,Yes,38753230,PositivelyAssociatedWith
3419,Alpha-ketoglutarate,Mus musculus,UM-HET3,20000 ppm food,18 months,until death,1.00,NS,-5.0,NS,Female,8.0,NS,Yes,38753230,NotAssociatedWith
3420,Alpha-ketoglutarate,Mus musculus,UM-HET3,20000 ppm food,18 months,until death,3.00,NS,-3.0,NS,Male,0.0,NS,Yes,38753230,NotAssociatedWith
3421,X203 (anti-IL-11),Mus musculus,C57BL/6 J,40 ppm bodyweight monthly intraperitoneal inje...,17.2 months,until death,25.00,S,NaN,NaN,Female,NaN,NaN,No,39020175,PositivelyAssociatedWith


In [49]:
# ── 2c. Resolve compound name → PubChem CID (case-insensitive) ───────────────
file_drugage['Head'] = file_drugage['compound_name'].str.lower().map(Pubchem_Syn_fil_dict_lower)
# Strip float suffix (e.g. 12345.0 → 12345) introduced by pandas
file_drugage['Head'] = file_drugage['Head'].astype(str).str.replace(r'\.0$', '', regex=True)


# Running dedup before lookup keeps an arbitrary first occurrence of each compound name
# even if it has no CID; running it after means we drop rows that couldn't be resolved.
file_drugage = file_drugage.drop_duplicates(subset=['compound_name'], keep='first', ignore_index=True)

# Drop rows where no PubChem CID could be resolved
before = len(file_drugage)
file_drugage = file_drugage[file_drugage['Head'] != 'nan']
print(f"CID resolved: {len(file_drugage):,} kept / {before - len(file_drugage):,} dropped")
file_drugage

CID resolved: 727 kept / 289 dropped


,compound_name,species,strain,dosage,age_at_initiation,treatment_duration,avg_lifespan_change_percent,avg_lifespan_significance,max_lifespan_change_percent,max_lifespan_significance,gender,weight_change_percent,weight_change_significance,ITP,pubmed_id,Relation,Head
0,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.0001 mM,NaN,NaN,2.30,NS,NaN,NaN,Male,NaN,NaN,No,98863,NotAssociatedWith,19463
4,Nitrosodimethylurea,Drosophila melanogaster,D32,0.00%,NaN,NaN,-37.00,S,-33.0,NaN,Male,NaN,NaN,No,119648,NegativelyAssociatedWith,83273
5,2-Ethyl-6-methylpyridin-3-ol hydrochloride,Drosophila melanogaster,R(1)2 yy/y+Y,0.01%,NaN,NaN,22.00,S,17.0,NaN,Male,NaN,NaN,No,119648,PositivelyAssociatedWith,128852
6,Phenobarbital,Rattus norvegicus,Sprague Dawley,2 mg/kg/week,NaN,NaN,-9.57,S,NaN,NaN,Female,NaN,NaN,No,132031,NegativelyAssociatedWith,4763
7,Nicotine,Rattus norvegicus,Sprague Dawley,2 mg/kg/week,NaN,NaN,-8.51,S,NaN,NaN,Female,NaN,NaN,No,132031,NegativelyAssociatedWith,89594
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010,Meclizine,Mus musculus,UM-HET3,800 ppm food,12 months,until death,3.00,S,1.0,NS,Female,-12.0,S,Yes,38041783,PositivelyAssociatedWith,4034
1011,Dimethyl fumarate,Mus musculus,UM-HET3,120 ppm food,16 months,until death,-1.00,NS,-2.0,NS,Female,-5.0,NS,Yes,38041783,NotAssociatedWith,637568
1012,4-phenylbutyrate,Mus musculus,UM-HET3,1000 ppm food,9 months,until death,-2.00,S,-6.0,NS,Female,4.0,NS,Yes,38041783,NegativelyAssociatedWith,22053264
1013,Sodium thiosulfate,Mus musculus,UM-HET3,10000 ppm food,6 months,until death,-1.00,NS,0.0,NS,Male,-3.0,NS,Yes,38753230,NotAssociatedWith,24477


In [50]:
# ── 2e. Annotate IUPAC name and SMILES ───────────────────────────────────────
file_drugage['Head_detail_name'] = file_drugage['Head'].map(Pubchem_IUPAC_CID_Dict)
file_drugage['Head_SMILE_name']  = file_drugage['Head'].map(Pubchem_CID_Smile_Dict)
file_drugage['Tail']             = 'GO:0007568'
file_drugage['Tail_detail_name'] = 'aging'
file_drugage

,compound_name,species,strain,dosage,age_at_initiation,treatment_duration,avg_lifespan_change_percent,avg_lifespan_significance,max_lifespan_change_percent,max_lifespan_significance,...,weight_change_percent,weight_change_significance,ITP,pubmed_id,Relation,Head,Head_detail_name,Head_SMILE_name,Tail,Tail_detail_name
0,Diethylhydroxylamine,Drosophila melanogaster,Oregon-R,0.0001 mM,NaN,NaN,2.30,NS,NaN,NaN,...,NaN,NaN,No,98863,NotAssociatedWith,19463,"N,N-diethylhydroxylamine",CCN(CC)O,GO:0007568,aging
4,Nitrosodimethylurea,Drosophila melanogaster,D32,0.00%,NaN,NaN,-37.00,S,-33.0,NaN,...,NaN,NaN,No,119648,NegativelyAssociatedWith,83273,"1,3-dimethyl-1-nitrosourea",CNC(=O)N(C)N=O,GO:0007568,aging
5,2-Ethyl-6-methylpyridin-3-ol hydrochloride,Drosophila melanogaster,R(1)2 yy/y+Y,0.01%,NaN,NaN,22.00,S,17.0,NaN,...,NaN,NaN,No,119648,PositivelyAssociatedWith,128852,2-ethyl-6-methylpyridin-3-ol;hydrochloride,CCC1=C(C=CC(=N1)C)O.Cl,GO:0007568,aging
6,Phenobarbital,Rattus norvegicus,Sprague Dawley,2 mg/kg/week,NaN,NaN,-9.57,S,NaN,NaN,...,NaN,NaN,No,132031,NegativelyAssociatedWith,4763,"5-ethyl-5-phenyl-1,3-diazinane-2,4,6-trione",CCC1(C(=O)NC(=O)NC1=O)C2=CC=CC=C2,GO:0007568,aging
7,Nicotine,Rattus norvegicus,Sprague Dawley,2 mg/kg/week,NaN,NaN,-8.51,S,NaN,NaN,...,NaN,NaN,No,132031,NegativelyAssociatedWith,89594,3-[(2S)-1-methylpyrrolidin-2-yl]pyridine,CN1CCC[C@H]1C2=CN=CC=C2,GO:0007568,aging
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010,Meclizine,Mus musculus,UM-HET3,800 ppm food,12 months,until death,3.00,S,1.0,NS,...,-12.0,S,Yes,38041783,PositivelyAssociatedWith,4034,1-[(4-chlorophenyl)-phenylmethyl]-4-[(3-methyl...,CC1=CC(=CC=C1)CN2CCN(CC2)C(C3=CC=CC=C3)C4=CC=C...,GO:0007568,aging
1011,Dimethyl fumarate,Mus musculus,UM-HET3,120 ppm food,16 months,until death,-1.00,NS,-2.0,NS,...,-5.0,NS,Yes,38041783,NotAssociatedWith,637568,dimethyl (E)-but-2-enedioate,COC(=O)/C=C/C(=O)OC,GO:0007568,aging
1012,4-phenylbutyrate,Mus musculus,UM-HET3,1000 ppm food,9 months,until death,-2.00,S,-6.0,NS,...,4.0,NS,Yes,38041783,NegativelyAssociatedWith,22053264,4-phenylbutanoate,C1=CC=C(C=C1)CCCC(=O)[O-],GO:0007568,aging
1013,Sodium thiosulfate,Mus musculus,UM-HET3,10000 ppm food,6 months,until death,-1.00,NS,0.0,NS,...,-3.0,NS,Yes,38753230,NotAssociatedWith,24477,disodium;dioxido-oxo-sulfanylidene-lambda6-sul...,[O-]S(=O)(=S)[O-].[Na+].[Na+],GO:0007568,aging


In [51]:
# ── 2f. Rename columns and build KG schema ────────────────────────────────────
file_drugage = file_drugage.rename(columns={'compound_name': 'Head_Alt_name', 'species': 'Species'})

file_drugage['Head_type']       = 'ChemicalEntity'
file_drugage['Tail_type']       = 'BiologicalProcess'       # updated to BiologicalProcess below
file_drugage['Relation_Source'] = 'DrugAge'
file_drugage['Data_type']       = 'Associative'

# Select columns; include provenance metadata from the source file
desired_order = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Tail_type',
    'Head_detail_name', 'Head_SMILE_name',
    'Relation_Source', 'Species', 'Head_Alt_name', 'Data_type',
    'dosage', 'age_at_initiation', 'treatment_duration', 'avg_lifespan_change_percent'
]
file_drugage = file_drugage[[c for c in desired_order if c in file_drugage.columns]]
file_drugage

,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Head_SMILE_name,Relation_Source,Species,Head_Alt_name,Data_type,dosage,age_at_initiation,treatment_duration,avg_lifespan_change_percent
0,19463,NotAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"N,N-diethylhydroxylamine",CCN(CC)O,DrugAge,Drosophila melanogaster,Diethylhydroxylamine,Associative,0.0001 mM,NaN,NaN,2.30
4,83273,NegativelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"1,3-dimethyl-1-nitrosourea",CNC(=O)N(C)N=O,DrugAge,Drosophila melanogaster,Nitrosodimethylurea,Associative,0.00%,NaN,NaN,-37.00
5,128852,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,2-ethyl-6-methylpyridin-3-ol;hydrochloride,CCC1=C(C=CC(=N1)C)O.Cl,DrugAge,Drosophila melanogaster,2-Ethyl-6-methylpyridin-3-ol hydrochloride,Associative,0.01%,NaN,NaN,22.00
6,4763,NegativelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"5-ethyl-5-phenyl-1,3-diazinane-2,4,6-trione",CCC1(C(=O)NC(=O)NC1=O)C2=CC=CC=C2,DrugAge,Rattus norvegicus,Phenobarbital,Associative,2 mg/kg/week,NaN,NaN,-9.57
7,89594,NegativelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,3-[(2S)-1-methylpyrrolidin-2-yl]pyridine,CN1CCC[C@H]1C2=CN=CC=C2,DrugAge,Rattus norvegicus,Nicotine,Associative,2 mg/kg/week,NaN,NaN,-8.51
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010,4034,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,1-[(4-chlorophenyl)-phenylmethyl]-4-[(3-methyl...,CC1=CC(=CC=C1)CN2CCN(CC2)C(C3=CC=CC=C3)C4=CC=C...,DrugAge,Mus musculus,Meclizine,Associative,800 ppm food,12 months,until death,3.00
1011,637568,NotAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,dimethyl (E)-but-2-enedioate,COC(=O)/C=C/C(=O)OC,DrugAge,Mus musculus,Dimethyl fumarate,Associative,120 ppm food,16 months,until death,-1.00
1012,22053264,NegativelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,4-phenylbutanoate,C1=CC=C(C=C1)CCCC(=O)[O-],DrugAge,Mus musculus,4-phenylbutyrate,Associative,1000 ppm food,9 months,until death,-2.00
1013,24477,NotAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,disodium;dioxido-oxo-sulfanylidene-lambda6-sul...,[O-]S(=O)(=S)[O-].[Na+].[Na+],DrugAge,Mus musculus,Sodium thiosulfate,Associative,10000 ppm food,6 months,until death,-1.00


In [52]:

print("F1 Relation distribution:")
print(file_drugage['Relation'].value_counts())
print("\nF1 Species distribution:")
print(file_drugage['Species'].value_counts())

F1 Relation distribution:
Relation
PositivelyAssociatedWith    407
NotAssociatedWith           253
NegativelyAssociatedWith     67
Name: count, dtype: int64

F1 Species distribution:
Species
Caenorhabditis elegans       404
Drosophila melanogaster      197
Mus musculus                  57
Rattus norvegicus             20
Saccharomyces cerevisiae      15
Brachionus manjavacas          7
Asplanchna brightwelli         5
Musca domestica                4
Adineta vaga                   3
Philodina acuticornis          3
Zaprionus paravittiger         3
Podospora anserina             2
Aedes aegypti                  1
Mesocricetus auratus           1
Musca Domestica                1
Caenorhabditis briggsae        1
Caenorhabditis tropicalis      1
Aedes albopictus               1
Mus Musculus                   1
Name: count, dtype: int64


In [56]:
# relation_label

# Saccharomyces cerevisiae 
file_drugage

,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Head_SMILE_name,Relation_Source,Species,Head_Alt_name,Data_type,dosage,age_at_initiation,treatment_duration,avg_lifespan_change_percent
0,19463,NotAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"N,N-diethylhydroxylamine",CCN(CC)O,DrugAge,Drosophila melanogaster,Diethylhydroxylamine,Associative,0.0001 mM,NaN,NaN,2.30
4,83273,NegativelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"1,3-dimethyl-1-nitrosourea",CNC(=O)N(C)N=O,DrugAge,Drosophila melanogaster,Nitrosodimethylurea,Associative,0.00%,NaN,NaN,-37.00
5,128852,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,2-ethyl-6-methylpyridin-3-ol;hydrochloride,CCC1=C(C=CC(=N1)C)O.Cl,DrugAge,Drosophila melanogaster,2-Ethyl-6-methylpyridin-3-ol hydrochloride,Associative,0.01%,NaN,NaN,22.00
6,4763,NegativelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"5-ethyl-5-phenyl-1,3-diazinane-2,4,6-trione",CCC1(C(=O)NC(=O)NC1=O)C2=CC=CC=C2,DrugAge,Rattus norvegicus,Phenobarbital,Associative,2 mg/kg/week,NaN,NaN,-9.57
7,89594,NegativelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,3-[(2S)-1-methylpyrrolidin-2-yl]pyridine,CN1CCC[C@H]1C2=CN=CC=C2,DrugAge,Rattus norvegicus,Nicotine,Associative,2 mg/kg/week,NaN,NaN,-8.51
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010,4034,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,1-[(4-chlorophenyl)-phenylmethyl]-4-[(3-methyl...,CC1=CC(=CC=C1)CN2CCN(CC2)C(C3=CC=CC=C3)C4=CC=C...,DrugAge,Mus musculus,Meclizine,Associative,800 ppm food,12 months,until death,3.00
1011,637568,NotAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,dimethyl (E)-but-2-enedioate,COC(=O)/C=C/C(=O)OC,DrugAge,Mus musculus,Dimethyl fumarate,Associative,120 ppm food,16 months,until death,-1.00
1012,22053264,NegativelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,4-phenylbutanoate,C1=CC=C(C=C1)CCCC(=O)[O-],DrugAge,Mus musculus,4-phenylbutyrate,Associative,1000 ppm food,9 months,until death,-2.00
1013,24477,NotAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,disodium;dioxido-oxo-sulfanylidene-lambda6-sul...,[O-]S(=O)(=S)[O-].[Na+].[Na+],DrugAge,Mus musculus,Sodium thiosulfate,Associative,10000 ppm food,6 months,until death,-1.00


In [57]:
# Helper function to save a species + relation slice to the intermediate folder
def save_intermediate(df, species, relation_label, tag):
    subset = df[(df['Species'] == species) & (df['Relation'] == relation_label)]
    path = os.path.join(OUT_INTER, f"{tag}.csv")
    subset.to_csv(path, index=False)
    print(f"  {tag}.csv  →  {len(subset):,} rows")
    return subset

POS = 'PositivelyAssociatedWith'
NEG = 'NegativelyAssociatedWith'
NOT = 'NotAssociatedWith'

print("F1 intermediate files:")

# ── Yeast ─────────────────────────────────────────────────────────────────────
Yeast_F1_Pos = save_intermediate(file_drugage, 'Saccharomyces cerevisiae', POS, 'Yeast_CE_Pos_BioPro_F1')
Yeast_F1_Neg = save_intermediate(file_drugage, 'Saccharomyces cerevisiae', NEG, 'Yeast_CE_Neg_BioPro_F1')
Yeast_F1_Not = save_intermediate(file_drugage, 'Saccharomyces cerevisiae', NOT, 'Yeast_CE_Not_BioPro_F1')

# ── C. elegans ────────────────────────────────────────────────────────────────
Cele_F1_Pos = save_intermediate(file_drugage, 'Caenorhabditis elegans', POS, 'Cele_CE_Pos_BioPro_F1')
Cele_F1_Neg = save_intermediate(file_drugage, 'Caenorhabditis elegans', NEG, 'Cele_CE_Neg_BioPro_F1')
Cele_F1_Not = save_intermediate(file_drugage, 'Caenorhabditis elegans', NOT, 'Cele_CE_Not_BioPro_F1')

# ── Drosophila ────────────────────────────────────────────────────────────────
Droso_F1_Pos = save_intermediate(file_drugage, 'Drosophila melanogaster', POS, 'Droso_CE_Pos_BioPro_F1')
Droso_F1_Neg = save_intermediate(file_drugage, 'Drosophila melanogaster', NEG, 'Droso_CE_Neg_BioPro_F1')
Droso_F1_Not = save_intermediate(file_drugage, 'Drosophila melanogaster', NOT, 'Droso_CE_Not_BioPro_F1')

# ── Mouse ─────────────────────────────────────────────────────────────────────
Mouse_F1_Pos = save_intermediate(file_drugage, 'Mus musculus', POS, 'Mouse_CE_Pos_BioPro_F1')
Mouse_F1_Neg = save_intermediate(file_drugage, 'Mus musculus', NEG, 'Mouse_CE_Neg_BioPro_F1')
Mouse_F1_Not = save_intermediate(file_drugage, 'Mus musculus', NOT, 'Mouse_CE_Not_BioPro_F1')

# # ── Rat ────────────────────────────────────────────WE dont need it ───────────────────────────
# Rat_F1_Pos = save_intermediate(file_drugage, 'Rattus norvegicus', POS, 'Rat_CE_Pos_BioPro_F1')
# Rat_F1_Neg = save_intermediate(file_drugage, 'Rattus norvegicus', NEG, 'Rat_CE_Neg_BioPro_F1')
# Rat_F1_Not = save_intermediate(file_drugage, 'Rattus norvegicus', NOT, 'Rat_CE_Not_BioPro_F1')

F1 intermediate files:
  Yeast_CE_Pos_BioPro_F1.csv  →  10 rows
  Yeast_CE_Neg_BioPro_F1.csv  →  0 rows
  Yeast_CE_Not_BioPro_F1.csv  →  5 rows
  Cele_CE_Pos_BioPro_F1.csv  →  281 rows
  Cele_CE_Neg_BioPro_F1.csv  →  39 rows
  Cele_CE_Not_BioPro_F1.csv  →  84 rows
  Droso_CE_Pos_BioPro_F1.csv  →  66 rows
  Droso_CE_Neg_BioPro_F1.csv  →  15 rows
  Droso_CE_Not_BioPro_F1.csv  →  116 rows
  Mouse_CE_Pos_BioPro_F1.csv  →  24 rows
  Mouse_CE_Neg_BioPro_F1.csv  →  3 rows
  Mouse_CE_Not_BioPro_F1.csv  →  30 rows


---
## 3 · File 2 (F2) Pipeline — `DrugAge Browse.csv`


In [34]:
# ── 3a. Load F2 raw data ──────────────────────────────────────────────────────
DA_file = pd.read_csv(DRUGAGE_F2_PATH)
print(f"F2 raw: {len(DA_file):,} rows")
print("Columns:", list(DA_file.columns))
print("Species:", DA_file['Species'].value_counts().to_dict())

DA_file

F2 raw: 3,270 rows
Columns: ['Compound/Formulation', 'Species', 'Avg/Med Lifespan Change (%)', 'Max Lifespan Change (%)', 'Strain', 'Dosage', 'Significant?', 'Reference']
Species: {'Caenorhabditis elegans': 1587, 'Drosophila melanogaster': 980, 'Mus musculus': 220, 'Saccharomyces cerevisiae': 79, 'Zaprionus paravittiger': 72, 'Rattus norvegicus': 60, 'Drosophila mojavensis': 52, 'Anastrepha ludens': 25, 'Asplanchna brightwelli': 23, 'Caenorhabditis briggsae': 14, 'Musca domestica': 14, 'Aedes aegypti': 13, 'Brachionus manjavacas': 12, 'Drosophila virilis': 12, 'Paramecium tetraurelia': 10, 'Philodina acuticornis': 10, 'Aedes albopictus': 9, 'Drosophila kikkawai': 8, 'Acheta domesticus': 8, 'Caenorhabditis tropicalis': 7, 'Nothobranchius guentheri': 7, 'Drosophila bipectinata': 6, 'Podospora anserina': 6, 'Aeolosoma viride': 5, 'Mytilina brevispina': 4, 'Adineta vaga': 4, 'Daphnia pulex clone TCO': 4, 'Nothobranchius furzeri': 3, 'Anopheles stephensi': 3, 'Mesocricetus auratus': 3, 'Mus

,Compound/Formulation,Species,Avg/Med Lifespan Change (%),Max Lifespan Change (%),Strain,Dosage,Significant?,Reference
0,Lithocholic acid,Saccharomyces cerevisiae,105.50,93.0,BY4742,50 µM,Y,30405886
1,Thioflavin T,Caenorhabditis elegans,60.00,78.0,N2,50-100 µM,Y,21451522
2,Bacitracin A,Caenorhabditis elegans,58.50,74.0,N2,0.01,Y,22114686
3,Acetaminophen,Caenorhabditis elegans,48.60,66.0,N2,0.10%,Y,22114686
4,Minocycline,Drosophila melanogaster,50.00,66.0,Oregon R,0.87 mM,Y,23185716
...,...,...,...,...,...,...,...,...
3265,Mifepristone,Drosophila melanogaster,-3.45,NaN,NaN,200 μg/mL,N,32648907
3266,Mifepristone,Drosophila melanogaster,15.20,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,Y,28451923
3267,Mifepristone,Drosophila melanogaster,56.35,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,Y,28451923
3268,Mifepristone,Drosophila melanogaster,67.10,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,Y,28451923


In [36]:
# ── F2a: Rename columns to match F1 schema ───────────────────────────────────
DA_file = DA_file.rename(columns={
    'Compound/Formulation'       : 'compound_name',
    'Avg/Med Lifespan Change (%)': 'avg_lifespan_change_percent',
    'Max Lifespan Change (%)'    : 'max_lifespan_change_percent',
    'Significant?'               : 'avg_lifespan_significance',
    'Reference'                  : 'pubmed_id'
})

# ── F2b: Standardise significance values Y/N → S/NS ──────────────────────────
DA_file['avg_lifespan_significance'] = DA_file['avg_lifespan_significance'].map(
    {'Y': 'S', 'N': 'NS'})

# Drop NaN significance — cannot make any claim
DA_file = DA_file[DA_file['avg_lifespan_significance'].notna()]
print(f"After dropping NaN significance: {len(DA_file):,} rows")

# ── F2c: Fix species name inconsistencies ─────────────────────────────────────
DA_file['Species'] = DA_file['Species'].str.strip()
DA_file['Species'] = DA_file['Species'].apply(
    lambda x: ' '.join([x.split()[0].capitalize(), x.split()[1].lower()])
    if len(x.split()) == 2 else x
)
DA_file

After dropping NaN significance: 3,078 rows


,compound_name,Species,avg_lifespan_change_percent,max_lifespan_change_percent,Strain,Dosage,avg_lifespan_significance,pubmed_id
0,Lithocholic acid,Saccharomyces cerevisiae,105.50,93.0,BY4742,50 µM,S,30405886
1,Thioflavin T,Caenorhabditis elegans,60.00,78.0,N2,50-100 µM,S,21451522
2,Bacitracin A,Caenorhabditis elegans,58.50,74.0,N2,0.01,S,22114686
3,Acetaminophen,Caenorhabditis elegans,48.60,66.0,N2,0.10%,S,22114686
4,Minocycline,Drosophila melanogaster,50.00,66.0,Oregon R,0.87 mM,S,23185716
...,...,...,...,...,...,...,...,...
3265,Mifepristone,Drosophila melanogaster,-3.45,NaN,NaN,200 μg/mL,NS,32648907
3266,Mifepristone,Drosophila melanogaster,15.20,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923
3267,Mifepristone,Drosophila melanogaster,56.35,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923
3268,Mifepristone,Drosophila melanogaster,67.10,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923


In [37]:
# ── F2d: Filter to target species ────────────────────────────────────────────
DA_file = DA_file[DA_file['Species'].isin(SPECIES_LIST)]
print(f"After species filter: {len(DA_file):,} rows")
print("Species kept:", DA_file['Species'].value_counts().to_dict())
DA_file

After species filter: 2,744 rows
Species kept: {'Caenorhabditis elegans': 1575, 'Drosophila melanogaster': 902, 'Mus musculus': 201, 'Saccharomyces cerevisiae': 66}


,compound_name,Species,avg_lifespan_change_percent,max_lifespan_change_percent,Strain,Dosage,avg_lifespan_significance,pubmed_id
0,Lithocholic acid,Saccharomyces cerevisiae,105.50,93.0,BY4742,50 µM,S,30405886
1,Thioflavin T,Caenorhabditis elegans,60.00,78.0,N2,50-100 µM,S,21451522
2,Bacitracin A,Caenorhabditis elegans,58.50,74.0,N2,0.01,S,22114686
3,Acetaminophen,Caenorhabditis elegans,48.60,66.0,N2,0.10%,S,22114686
4,Minocycline,Drosophila melanogaster,50.00,66.0,Oregon R,0.87 mM,S,23185716
...,...,...,...,...,...,...,...,...
3265,Mifepristone,Drosophila melanogaster,-3.45,NaN,NaN,200 μg/mL,NS,32648907
3266,Mifepristone,Drosophila melanogaster,15.20,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923
3267,Mifepristone,Drosophila melanogaster,56.35,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923
3268,Mifepristone,Drosophila melanogaster,67.10,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923


In [39]:
# ── F2e: Drop NaN lifespan change ────────────────────────────────────────────
DA_file = DA_file[~DA_file['avg_lifespan_change_percent'].isna()]
print(f"After dropping NaN lifespan change: {len(DA_file):,} rows")


After dropping NaN lifespan change: 2,701 rows


In [40]:
# ── F2f: Resolve compound name → PubChem CID ─────────────────────────────────
DA_file['Head'] = DA_file['compound_name'].str.lower().map(Pubchem_Syn_fil_dict_lower)
DA_file['Head'] = DA_file['Head'].astype(str).str.replace(r'\.0$', '', regex=True)
before = len(DA_file)
DA_file = DA_file[DA_file['Head'] != 'nan']
print(f"CID resolved: {len(DA_file):,} kept / {before - len(DA_file):,} dropped")
DA_file

CID resolved: 2,140 kept / 561 dropped


/tmp/ipykernel_2371907/2529179731.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DA_file['Head'] = DA_file['compound_name'].str.lower().map(Pubchem_Syn_fil_dict_lower)
/tmp/ipykernel_2371907/2529179731.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DA_file['Head'] = DA_file['Head'].astype(str).str.replace(r'\.0$', '', regex=True)


,compound_name,Species,avg_lifespan_change_percent,max_lifespan_change_percent,Strain,Dosage,avg_lifespan_significance,pubmed_id,Head
0,Lithocholic acid,Saccharomyces cerevisiae,105.50,93.0,BY4742,50 µM,S,30405886,9903
1,Thioflavin T,Caenorhabditis elegans,60.00,78.0,N2,50-100 µM,S,21451522,16953
2,Bacitracin A,Caenorhabditis elegans,58.50,74.0,N2,0.01,S,22114686,60196264
3,Acetaminophen,Caenorhabditis elegans,48.60,66.0,N2,0.10%,S,22114686,1983
4,Minocycline,Drosophila melanogaster,50.00,66.0,Oregon R,0.87 mM,S,23185716,54675783
...,...,...,...,...,...,...,...,...,...
3265,Mifepristone,Drosophila melanogaster,-3.45,NaN,NaN,200 μg/mL,NS,32648907,55245
3266,Mifepristone,Drosophila melanogaster,15.20,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245
3267,Mifepristone,Drosophila melanogaster,56.35,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245
3268,Mifepristone,Drosophila melanogaster,67.10,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245


In [43]:
# ── F2g: Assign aging direction using same groupby logic as F1 ────────────────
def classify_compound(group):
    sig_rows = group[group['avg_lifespan_significance'] == 'S']

    # No significant rows at all → NotAssociated for entire group
    if len(sig_rows) == 0:
        group['Relation'] = 'NotAssociatedWith'
        return group

    # Has significant rows → assign direction per row
    def assign_direction(row):
        if row['avg_lifespan_significance'] != 'S':
            return 'NotAssociatedWith'
        pct = row['avg_lifespan_change_percent']
        if pct > 1:
            return 'PositivelyAssociatedWith'
        elif pct < 1:
            return 'NegativelyAssociatedWith'
        else:
            return 'NotAssociatedWith'

    group['Relation'] = group.apply(assign_direction, axis=1)
    return group

DA_file = (
    DA_file
    .groupby(['compound_name', 'Species'], group_keys=False)
    .apply(classify_compound)
)
print("Relation distribution:", DA_file['Relation'].value_counts().to_dict())
DA_file

Relation distribution: {'PositivelyAssociatedWith': 1134, 'NotAssociatedWith': 731, 'NegativelyAssociatedWith': 275}


/tmp/ipykernel_2371907/833360082.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(classify_compound)


,compound_name,Species,avg_lifespan_change_percent,max_lifespan_change_percent,Strain,Dosage,avg_lifespan_significance,pubmed_id,Head,Tail,Relation
0,Lithocholic acid,Saccharomyces cerevisiae,105.50,93.0,BY4742,50 µM,S,30405886,9903,PositivelyAssociatedWithAging,PositivelyAssociatedWith
1,Thioflavin T,Caenorhabditis elegans,60.00,78.0,N2,50-100 µM,S,21451522,16953,PositivelyAssociatedWithAging,PositivelyAssociatedWith
2,Bacitracin A,Caenorhabditis elegans,58.50,74.0,N2,0.01,S,22114686,60196264,PositivelyAssociatedWithAging,PositivelyAssociatedWith
3,Acetaminophen,Caenorhabditis elegans,48.60,66.0,N2,0.10%,S,22114686,1983,PositivelyAssociatedWithAging,PositivelyAssociatedWith
4,Minocycline,Drosophila melanogaster,50.00,66.0,Oregon R,0.87 mM,S,23185716,54675783,PositivelyAssociatedWithAging,PositivelyAssociatedWith
...,...,...,...,...,...,...,...,...,...,...,...
3265,Mifepristone,Drosophila melanogaster,-3.45,NaN,NaN,200 μg/mL,NS,32648907,55245,NotAssociatedWithAging,NotAssociatedWith
3266,Mifepristone,Drosophila melanogaster,15.20,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245,PositivelyAssociatedWithAging,PositivelyAssociatedWith
3267,Mifepristone,Drosophila melanogaster,56.35,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245,PositivelyAssociatedWithAging,PositivelyAssociatedWith
3268,Mifepristone,Drosophila melanogaster,67.10,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245,PositivelyAssociatedWithAging,PositivelyAssociatedWith


In [58]:
DA_file['Tail']             = 'GO:0007568'
DA_file['Tail_detail_name'] = 'aging'
DA_file

,compound_name,Species,avg_lifespan_change_percent,max_lifespan_change_percent,Strain,Dosage,avg_lifespan_significance,pubmed_id,Head,Tail,Relation,Tail_detail_name
0,Lithocholic acid,Saccharomyces cerevisiae,105.50,93.0,BY4742,50 µM,S,30405886,9903,GO:0007568,PositivelyAssociatedWith,aging
1,Thioflavin T,Caenorhabditis elegans,60.00,78.0,N2,50-100 µM,S,21451522,16953,GO:0007568,PositivelyAssociatedWith,aging
2,Bacitracin A,Caenorhabditis elegans,58.50,74.0,N2,0.01,S,22114686,60196264,GO:0007568,PositivelyAssociatedWith,aging
3,Acetaminophen,Caenorhabditis elegans,48.60,66.0,N2,0.10%,S,22114686,1983,GO:0007568,PositivelyAssociatedWith,aging
4,Minocycline,Drosophila melanogaster,50.00,66.0,Oregon R,0.87 mM,S,23185716,54675783,GO:0007568,PositivelyAssociatedWith,aging
...,...,...,...,...,...,...,...,...,...,...,...,...
3265,Mifepristone,Drosophila melanogaster,-3.45,NaN,NaN,200 μg/mL,NS,32648907,55245,GO:0007568,NotAssociatedWith,aging
3266,Mifepristone,Drosophila melanogaster,15.20,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245,GO:0007568,PositivelyAssociatedWith,aging
3267,Mifepristone,Drosophila melanogaster,56.35,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245,GO:0007568,PositivelyAssociatedWith,aging
3268,Mifepristone,Drosophila melanogaster,67.10,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245,GO:0007568,PositivelyAssociatedWith,aging


In [61]:
# ── F2h: Annotate IUPAC name and SMILES ──────────────────────────────────────
DA_file['Head_detail_name'] = DA_file['Head'].map(Pubchem_IUPAC_CID_Dict)
DA_file['Head_SMILE_name']  = DA_file['Head'].map(Pubchem_CID_Smile_Dict)

# ── F2i: Build KG schema ─────────────────────────────────────────────────────
DA_file = DA_file.rename(columns={'compound_name': 'Head_Alt_name'})
DA_file['Head_type']       = 'ChemicalEntity'
DA_file['Tail_type']       = 'BiologicalProcess'
DA_file['Relation_Source'] = 'DrugAge'
DA_file['Data_type']       = 'Associative'

desired_order = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Tail_type',
    'Head_detail_name', 'Head_SMILE_name',
    'Relation_Source', 'Species', 'Head_Alt_name', 'Data_type',
    'Dosage', 'avg_lifespan_change_percent', 'pubmed_id'
]
DA_file

,Head_Alt_name,Species,avg_lifespan_change_percent,max_lifespan_change_percent,Strain,Dosage,avg_lifespan_significance,pubmed_id,Head,Tail,Relation,Tail_detail_name,Head_detail_name,Head_SMILE_name,Head_type,Tail_type,Relation_Source,Data_type
0,Lithocholic acid,Saccharomyces cerevisiae,105.50,93.0,BY4742,50 µM,S,30405886,9903,GO:0007568,PositivelyAssociatedWith,aging,"(4R)-4-[(3R,5R,8R,9S,10S,13R,14S,17R)-3-hydrox...",C[C@H](CCC(=O)O)[C@H]1CC[C@@H]2[C@@]1(CC[C@H]3...,ChemicalEntity,BiologicalProcess,DrugAge,Associative
1,Thioflavin T,Caenorhabditis elegans,60.00,78.0,N2,50-100 µM,S,21451522,16953,GO:0007568,PositivelyAssociatedWith,aging,"4-(3,6-dimethyl-1,3-benzothiazol-3-ium-2-yl)-N...",CC1=CC2=C(C=C1)[N+](=C(S2)C3=CC=C(C=C3)N(C)C)C...,ChemicalEntity,BiologicalProcess,DrugAge,Associative
2,Bacitracin A,Caenorhabditis elegans,58.50,74.0,N2,0.01,S,22114686,60196264,GO:0007568,PositivelyAssociatedWith,aging,"(4S)-4-[[(2S)-2-[[(4R)-2-[(1R,2R)-1-amino-2-me...",CC[C@@H](C)[C@@H]1C(=O)N[C@@H](C(=O)N[C@@H](C(...,ChemicalEntity,BiologicalProcess,DrugAge,Associative
3,Acetaminophen,Caenorhabditis elegans,48.60,66.0,N2,0.10%,S,22114686,1983,GO:0007568,PositivelyAssociatedWith,aging,N-(4-hydroxyphenyl)acetamide,CC(=O)NC1=CC=C(C=C1)O,ChemicalEntity,BiologicalProcess,DrugAge,Associative
4,Minocycline,Drosophila melanogaster,50.00,66.0,Oregon R,0.87 mM,S,23185716,54675783,GO:0007568,PositivelyAssociatedWith,aging,"(4S,4aS,5aR,12aR)-4,7-bis(dimethylamino)-1,10,...",CN(C)[C@H]1[C@@H]2C[C@@H]3CC4=C(C=CC(=C4C(=C3C...,ChemicalEntity,BiologicalProcess,DrugAge,Associative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3265,Mifepristone,Drosophila melanogaster,-3.45,NaN,NaN,200 μg/mL,NS,32648907,55245,GO:0007568,NotAssociatedWith,aging,"(8S,11R,13S,14S,17S)-11-[4-(dimethylamino)phen...",CC#C[C@@]1(CC[C@@H]2[C@@]1(C[C@@H](C3=C4CCC(=O...,ChemicalEntity,BiologicalProcess,DrugAge,Associative
3266,Mifepristone,Drosophila melanogaster,15.20,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245,GO:0007568,PositivelyAssociatedWith,aging,"(8S,11R,13S,14S,17S)-11-[4-(dimethylamino)phen...",CC#C[C@@]1(CC[C@@H]2[C@@]1(C[C@@H](C3=C4CCC(=O...,ChemicalEntity,BiologicalProcess,DrugAge,Associative
3267,Mifepristone,Drosophila melanogaster,56.35,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245,GO:0007568,PositivelyAssociatedWith,aging,"(8S,11R,13S,14S,17S)-11-[4-(dimethylamino)phen...",CC#C[C@@]1(CC[C@@H]2[C@@]1(C[C@@H](C3=C4CCC(=O...,ChemicalEntity,BiologicalProcess,DrugAge,Associative
3268,Mifepristone,Drosophila melanogaster,67.10,NaN,Progeny of w[1118] X y; ElavGS,160 μg/mL,S,28451923,55245,GO:0007568,PositivelyAssociatedWith,aging,"(8S,11R,13S,14S,17S)-11-[4-(dimethylamino)phen...",CC#C[C@@]1(CC[C@@H]2[C@@]1(C[C@@H](C3=C4CCC(=O...,ChemicalEntity,BiologicalProcess,DrugAge,Associative


In [63]:
DA_file = DA_file[[c for c in desired_order if c in DA_file.columns]]
DA_file

,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Head_SMILE_name,Relation_Source,Species,Head_Alt_name,Data_type,Dosage,avg_lifespan_change_percent,pubmed_id
0,9903,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"(4R)-4-[(3R,5R,8R,9S,10S,13R,14S,17R)-3-hydrox...",C[C@H](CCC(=O)O)[C@H]1CC[C@@H]2[C@@]1(CC[C@H]3...,DrugAge,Saccharomyces cerevisiae,Lithocholic acid,Associative,50 µM,105.50,30405886
1,16953,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"4-(3,6-dimethyl-1,3-benzothiazol-3-ium-2-yl)-N...",CC1=CC2=C(C=C1)[N+](=C(S2)C3=CC=C(C=C3)N(C)C)C...,DrugAge,Caenorhabditis elegans,Thioflavin T,Associative,50-100 µM,60.00,21451522
2,60196264,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"(4S)-4-[[(2S)-2-[[(4R)-2-[(1R,2R)-1-amino-2-me...",CC[C@@H](C)[C@@H]1C(=O)N[C@@H](C(=O)N[C@@H](C(...,DrugAge,Caenorhabditis elegans,Bacitracin A,Associative,0.01,58.50,22114686
3,1983,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,N-(4-hydroxyphenyl)acetamide,CC(=O)NC1=CC=C(C=C1)O,DrugAge,Caenorhabditis elegans,Acetaminophen,Associative,0.10%,48.60,22114686
4,54675783,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"(4S,4aS,5aR,12aR)-4,7-bis(dimethylamino)-1,10,...",CN(C)[C@H]1[C@@H]2C[C@@H]3CC4=C(C=CC(=C4C(=C3C...,DrugAge,Drosophila melanogaster,Minocycline,Associative,0.87 mM,50.00,23185716
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3265,55245,NotAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"(8S,11R,13S,14S,17S)-11-[4-(dimethylamino)phen...",CC#C[C@@]1(CC[C@@H]2[C@@]1(C[C@@H](C3=C4CCC(=O...,DrugAge,Drosophila melanogaster,Mifepristone,Associative,200 μg/mL,-3.45,32648907
3266,55245,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"(8S,11R,13S,14S,17S)-11-[4-(dimethylamino)phen...",CC#C[C@@]1(CC[C@@H]2[C@@]1(C[C@@H](C3=C4CCC(=O...,DrugAge,Drosophila melanogaster,Mifepristone,Associative,160 μg/mL,15.20,28451923
3267,55245,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"(8S,11R,13S,14S,17S)-11-[4-(dimethylamino)phen...",CC#C[C@@]1(CC[C@@H]2[C@@]1(C[C@@H](C3=C4CCC(=O...,DrugAge,Drosophila melanogaster,Mifepristone,Associative,160 μg/mL,56.35,28451923
3268,55245,PositivelyAssociatedWith,GO:0007568,ChemicalEntity,BiologicalProcess,"(8S,11R,13S,14S,17S)-11-[4-(dimethylamino)phen...",CC#C[C@@]1(CC[C@@H]2[C@@]1(C[C@@H](C3=C4CCC(=O...,DrugAge,Drosophila melanogaster,Mifepristone,Associative,160 μg/mL,67.10,28451923


In [64]:
print("F2 Relation distribution:")
print(DA_file['Relation'].value_counts())
print("\nF2 Species distribution:")
print(DA_file['Species'].value_counts())

F2 Relation distribution:
Relation
PositivelyAssociatedWith    1134
NotAssociatedWith            731
NegativelyAssociatedWith     275
Name: count, dtype: int64

F2 Species distribution:
Species
Caenorhabditis elegans      1246
Drosophila melanogaster      692
Mus musculus                 152
Saccharomyces cerevisiae      50
Name: count, dtype: int64


In [66]:
# ── F2k: Save intermediate files ─────────────────────────────────────────────
print("\nF2 intermediate files:")

Yeast_F2_Pos = save_intermediate(DA_file, 'Saccharomyces cerevisiae', POS, 'Yeast_CE_Pos_BioPro_F2')
Yeast_F2_Neg = save_intermediate(DA_file, 'Saccharomyces cerevisiae', NEG, 'Yeast_CE_Neg_BioPro_F2')
Yeast_F2_Not = save_intermediate(DA_file, 'Saccharomyces cerevisiae', NOT, 'Yeast_CE_Not_BioPro_F2')

Cele_F2_Pos  = save_intermediate(DA_file, 'Caenorhabditis elegans',   POS, 'Cele_CE_Pos_BioPro_F2')
Cele_F2_Neg  = save_intermediate(DA_file, 'Caenorhabditis elegans',   NEG, 'Cele_CE_Neg_BioPro_F2')
Cele_F2_Not  = save_intermediate(DA_file, 'Caenorhabditis elegans',   NOT, 'Cele_CE_Not_BioPro_F2')

Droso_F2_Pos = save_intermediate(DA_file, 'Drosophila melanogaster',  POS, 'Droso_CE_Pos_BioPro_F2')
Droso_F2_Neg = save_intermediate(DA_file, 'Drosophila melanogaster',  NEG, 'Droso_CE_Neg_BioPro_F2')
Droso_F2_Not = save_intermediate(DA_file, 'Drosophila melanogaster',  NOT, 'Droso_CE_Not_BioPro_F2')

Mouse_F2_Pos = save_intermediate(DA_file, 'Mus musculus',             POS, 'Mouse_CE_Pos_BioPro_F2')
Mouse_F2_Neg = save_intermediate(DA_file, 'Mus musculus',             NEG, 'Mouse_CE_Neg_BioPro_F2')
Mouse_F2_Not = save_intermediate(DA_file, 'Mus musculus',             NOT, 'Mouse_CE_Not_BioPro_F2')



F2 intermediate files:
  Yeast_CE_Pos_BioPro_F2.csv  →  36 rows
  Yeast_CE_Neg_BioPro_F2.csv  →  1 rows
  Yeast_CE_Not_BioPro_F2.csv  →  13 rows
  Cele_CE_Pos_BioPro_F2.csv  →  784 rows
  Cele_CE_Neg_BioPro_F2.csv  →  161 rows
  Cele_CE_Not_BioPro_F2.csv  →  301 rows
  Droso_CE_Pos_BioPro_F2.csv  →  242 rows
  Droso_CE_Neg_BioPro_F2.csv  →  106 rows
  Droso_CE_Not_BioPro_F2.csv  →  344 rows
  Mouse_CE_Pos_BioPro_F2.csv  →  72 rows
  Mouse_CE_Neg_BioPro_F2.csv  →  7 rows
  Mouse_CE_Not_BioPro_F2.csv  →  73 rows
  Rat_CE_Pos_BioPro_F2.csv  →  0 rows
  Rat_CE_Neg_BioPro_F2.csv  →  0 rows
  Rat_CE_Not_BioPro_F2.csv  →  0 rows


In [68]:
# List all final output CSVs (excluding intermediate folder)
print(f"Final output files under: {OUT_PATH}\n")
total = 0
for root, dirs, files in os.walk(OUT_PATH):
    # Skip the intermediate folder in the summary
    dirs[:] = [d for d in sorted(dirs) if d != 'drugage_intermediate']
    for fname in sorted(files):
        if fname.endswith('.csv') and 'DrugAge' in fname:
            full = os.path.join(root, fname)
            rows = sum(1 for _ in open(full)) - 1
            size = os.path.getsize(full)
            rel  = os.path.relpath(full, OUT_PATH)
            total += 1
            print(f"  {rel:<65s}  {rows:>5,} rows  {size:>8,} bytes")
print(f"\nTotal: {total} final files")

Final output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/drugage

  Cele/Cele_DrugAge_CE_inhibits_BIoP.csv                               648 rows   163,160 bytes
  Cele/Cele_DrugAge_CE_promotes_BIoP.csv                               131 rows    32,409 bytes
  Droso/Droso_DrugAge_CE_inhibits_BIoP.csv                             168 rows    43,096 bytes
  Droso/Droso_DrugAge_CE_promotes_BIoP.csv                              59 rows    13,645 bytes
  Mouse/Mouse_DrugAge_CE_inhibits_BIoP.csv                              50 rows    12,203 bytes
  Mouse/Mouse_DrugAge_CE_promotes_BIoP.csv                              12 rows     2,638 bytes
  Yeast/Yeast_DrugAge_CE_inhibits_BIoP.csv                              24 rows     7,001 bytes

Total: 7 final files


In [70]:
# drugage_intermediate
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/drugage'

In [73]:
! ls /storage/Arushi/090526_EvoAge/kg_formation/processed_data/drugage/drugage_intermediate

Cele_CE_Neg_BioPro_F1.csv   Mouse_CE_Neg_BioPro_F1.csv
Cele_CE_Neg_BioPro_F2.csv   Mouse_CE_Neg_BioPro_F2.csv
Cele_CE_Not_BioPro_F1.csv   Mouse_CE_Not_BioPro_F1.csv
Cele_CE_Not_BioPro_F2.csv   Mouse_CE_Not_BioPro_F2.csv
Cele_CE_Pos_BioPro_F1.csv   Mouse_CE_Pos_BioPro_F1.csv
Cele_CE_Pos_BioPro_F2.csv   Mouse_CE_Pos_BioPro_F2.csv
Droso_CE_Neg_BioPro_F1.csv  Yeast_CE_Neg_BioPro_F1.csv
Droso_CE_Neg_BioPro_F2.csv  Yeast_CE_Neg_BioPro_F2.csv
Droso_CE_Not_BioPro_F1.csv  Yeast_CE_Not_BioPro_F1.csv
Droso_CE_Not_BioPro_F2.csv  Yeast_CE_Not_BioPro_F2.csv
Droso_CE_Pos_BioPro_F1.csv  Yeast_CE_Pos_BioPro_F1.csv
Droso_CE_Pos_BioPro_F2.csv  Yeast_CE_Pos_BioPro_F2.csv


In [78]:
import os
import pandas as pd

INTER = OUT_PATH
OUT_MERGED = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/drugage/drugage_final'
os.makedirs(OUT_MERGED, exist_ok=True)

# ── Define pairs to merge (F1 + F2 → merged) ─────────────────────────────────
merge_pairs = [
    # C. elegans
    ('Cele_CE_Pos_BioPro_F1.csv',   'Cele_CE_Pos_BioPro_F2.csv',   'Cele_CE_Pos_BioPro_merged.csv'),
    ('Cele_CE_Neg_BioPro_F1.csv',   'Cele_CE_Neg_BioPro_F2.csv',   'Cele_CE_Neg_BioPro_merged.csv'),
    ('Cele_CE_Not_BioPro_F1.csv',   'Cele_CE_Not_BioPro_F2.csv',   'Cele_CE_Not_BioPro_merged.csv'),
    # Drosophila
    ('Droso_CE_Pos_BioPro_F1.csv',  'Droso_CE_Pos_BioPro_F2.csv',  'Droso_CE_Pos_BioPro_merged.csv'),
    ('Droso_CE_Neg_BioPro_F1.csv',  'Droso_CE_Neg_BioPro_F2.csv',  'Droso_CE_Neg_BioPro_merged.csv'),
    ('Droso_CE_Not_BioPro_F1.csv',  'Droso_CE_Not_BioPro_F2.csv',  'Droso_CE_Not_BioPro_merged.csv'),
    # Mouse
    ('Mouse_CE_Pos_BioPro_F1.csv',  'Mouse_CE_Pos_BioPro_F2.csv',  'Mouse_CE_Pos_BioPro_merged.csv'),
    ('Mouse_CE_Neg_BioPro_F1.csv',  'Mouse_CE_Neg_BioPro_F2.csv',  'Mouse_CE_Neg_BioPro_merged.csv'),
    ('Mouse_CE_Not_BioPro_F1.csv',  'Mouse_CE_Not_BioPro_F2.csv',  'Mouse_CE_Not_BioPro_merged.csv'),
    # Yeast
    ('Yeast_CE_Pos_BioPro_F1.csv',  'Yeast_CE_Pos_BioPro_F2.csv',  'Yeast_CE_Pos_BioPro_merged.csv'),
    ('Yeast_CE_Neg_BioPro_F1.csv',  'Yeast_CE_Neg_BioPro_F2.csv',  'Yeast_CE_Neg_BioPro_merged.csv'),
    ('Yeast_CE_Not_BioPro_F1.csv',  'Yeast_CE_Not_BioPro_F2.csv',  'Yeast_CE_Not_BioPro_merged.csv'),
]

# ── Merge each pair ───────────────────────────────────────────────────────────
print("Merging F1 + F2 intermediate files:")
for f1_name, f2_name, out_name in merge_pairs:
    f1_path  = os.path.join(INTER, 'drugage_intermediate', f1_name)
    f2_path  = os.path.join(INTER, 'drugage_intermediate', f2_name)
    out_path = os.path.join(OUT_MERGED, out_name)

    f1 = pd.read_csv(f1_path) if os.path.exists(f1_path) else pd.DataFrame()
    f2 = pd.read_csv(f2_path) if os.path.exists(f2_path) else pd.DataFrame()

    merged = pd.concat([f1, f2], ignore_index=True).drop_duplicates(
    subset=['Head', 'Relation', 'Tail'], 
    keep='first'
)

    merged.to_csv(out_path, index=False)
    print(f"  {out_name:45s} → F1: {len(f1):>4,} + F2: {len(f2):>4,} = merged: {len(merged):>4,} rows")

Merging F1 + F2 intermediate files:
  Cele_CE_Pos_BioPro_merged.csv                 → F1:  281 + F2:  784 = merged:  362 rows
  Cele_CE_Neg_BioPro_merged.csv                 → F1:   39 + F2:  161 = merged:   98 rows
  Cele_CE_Not_BioPro_merged.csv                 → F1:   84 + F2:  301 = merged:  199 rows
  Droso_CE_Pos_BioPro_merged.csv                → F1:   66 + F2:  242 = merged:  119 rows
  Droso_CE_Neg_BioPro_merged.csv                → F1:   15 + F2:  106 = merged:   52 rows
  Droso_CE_Not_BioPro_merged.csv                → F1:  116 + F2:  344 = merged:  165 rows
  Mouse_CE_Pos_BioPro_merged.csv                → F1:   24 + F2:   72 = merged:   41 rows
  Mouse_CE_Neg_BioPro_merged.csv                → F1:    3 + F2:    7 = merged:    9 rows
  Mouse_CE_Not_BioPro_merged.csv                → F1:   30 + F2:   73 = merged:   61 rows
  Yeast_CE_Pos_BioPro_merged.csv                → F1:   10 + F2:   36 = merged:   17 rows
  Yeast_CE_Neg_BioPro_merged.csv                → F1:    0 + F2:

/tmp/ipykernel_2371907/810509811.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = pd.concat([f1, f2], ignore_index=True).drop_duplicates(
